# Fleet Maintenance Planning

**Team:** 38

### 1. Project Overview

#### Objective

Predict which vehicles should be inspected and determine
the appropriate spare-parts budget for each inspected vehicle.

The objective is to minimise the total maintenance cost
using the competition's specified cost function.

#### Project Workflow

1. Data loading and exploration
2. Preprocessing and feature selection
3. Failure prediction
4. Repair-cost prediction
5. Cost-based decision optimisation
6. Final submission generation

### 2. Imports and Config

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Locate the project root without changing the working directory.
current = Path.cwd().resolve()

ROOT = next(
    (
        p for p in [current, *current.parents]
        if (p / "data").is_dir() and (p / "src").is_dir()
    ),
    None,
)

if ROOT is None:
    raise FileNotFoundError(
        "Could not locate project root containing data/ and src/."
    )

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Import all shared paths and settings from one place.
from src.config import (
    DATA_DIR,
    RESULTS_DIR,
    SUBMISSIONS_DIR,
    RANDOM_STATE,
)

print("Project root:", ROOT)
print("Data directory:", DATA_DIR)
print("Results directory:", RESULTS_DIR)

Project root: C:\Users\Avnish\Documents\GitHub\fleet_maintenance_p2_team38
Data directory: C:\Users\Avnish\Documents\GitHub\fleet_maintenance_p2_team38\data
Results directory: C:\Users\Avnish\Documents\GitHub\fleet_maintenance_p2_team38\results


### 3. Data Loading:

In [2]:
from src.feature_analysis import load_data

# Load training and test datasets
train_df, test_df = load_data(DATA_DIR)

# Load sample submission for final format validation
sample_submission = pd.read_csv(
    DATA_DIR / "sample_submission.csv"
)

print("Training data:", train_df.shape)
print("Testing data:", test_df.shape)
print("Sample submission:", sample_submission.shape)

Training data: (56000, 174)
Testing data: (20000, 171)
Sample submission: (20000, 3)


### 4. Data Validation

In [3]:
# Check required columns
assert "failure" in train_df.columns
assert "repair_cost" in train_df.columns
assert "id" in train_df.columns

# Check identifiers
assert train_df["id"].is_unique
assert test_df["id"].is_unique

# Check target
assert train_df["failure"].notna().all()
assert train_df["failure"].isin([0, 1]).all()

# Check feature compatibility
sensor_cols = [
    col for col in train_df.columns
    if col.startswith("s")
]

assert set(sensor_cols) == {
    col for col in test_df.columns
    if col.startswith("s")
}

print("Basic data validation passed!")

Basic data validation passed!


### 5. EDA Summary

- The failure rate is highly imbalanced (0.71%).
- Repair costs are strongly right-skewed.
- Sensor features contain substantial missing values.
- Eight sensors have more than 50% missing values.
- One sensor (`s146`) is constant.
- Several sensors show substantial train–test differences, suggesting potential distribution shift between depots.

### 6. Cross-Validation Setup

In [4]:
from src.cv_utils import load_folds, validate_folds
from src.lightgbm_model import get_base_features

# Select features available and usable at test time
features = get_base_features(train_df, test_df)

X = train_df[features]
y = train_df["failure"]

# Load and validate the shared CV folds
folds = load_folds()
validate_folds(folds, len(X))

print(f"Training vehicles: {len(X):,}")
print(f"Features selected: {len(features)}")
print(f"CV folds: {len(folds)}")
print(f"Features excluded: {train_df.shape[1] - len(features) - 3}")

Fold validation successful.
Training vehicles: 56,000
Features selected: 167
CV folds: 5
Features excluded: 4


In [5]:
from src.repair_model import generate_oof_log_repair_costs

oof_log_budgets = generate_oof_log_repair_costs(
    train_df=train_df,
    features=features,
    folds=folds,
)

print("Shape:", oof_log_budgets.shape)
print("All finite:", np.isfinite(oof_log_budgets).all())
print("Minimum:", oof_log_budgets.min())
print("Maximum:", oof_log_budgets.max())
print("Mean:", oof_log_budgets.mean())

V3 log repair-cost OOF fold 1/5
V3 log repair-cost OOF fold 2/5
V3 log repair-cost OOF fold 3/5
V3 log repair-cost OOF fold 4/5
V3 log repair-cost OOF fold 5/5
Shape: (56000,)
All finite: True
Minimum: 3.4944926258919082
Maximum: 2582.14107242053
Mean: 209.11082362818053


### 7. Out-of-Fold Predictions

In [6]:
from src.lightgbm_model import build_final_lgbm, run_cv
from src.repair_model import generate_oof_repair_budgets

# 1. Generate out-of-fold failure probabilities
failure_cv = run_cv(
    model=build_final_lgbm(),
    X=X,
    y=y,
    folds=folds,
    name="Final 5-seed LightGBM",
    verbose=True,
)

oof_failure_probabilities = failure_cv["oof"]

# 2. Generate out-of-fold repair-budget predictions
oof_repair_budgets = generate_oof_repair_budgets(
    train_df=train_df,
    features=features,
    folds=folds,
)

# Validate OOF predictions
assert len(oof_failure_probabilities) == len(train_df)
assert len(oof_repair_budgets) == len(train_df)
assert np.isfinite(oof_failure_probabilities).all()
assert np.isfinite(oof_repair_budgets).all()

print("\nOOF prediction generation complete.")
print("Failure probabilities:", oof_failure_probabilities.shape)
print("Repair budgets:", oof_repair_budgets.shape)

Fold validation successful.
Final 5-seed LightGBM | fold 1: ROC-AUC 0.9841 | PR-AUC 0.6715 | 24.7s
Final 5-seed LightGBM | fold 2: ROC-AUC 0.9832 | PR-AUC 0.5601 | 21.6s
Final 5-seed LightGBM | fold 3: ROC-AUC 0.9802 | PR-AUC 0.5186 | 21.7s
Final 5-seed LightGBM | fold 4: ROC-AUC 0.9900 | PR-AUC 0.4464 | 22.4s
Final 5-seed LightGBM | fold 5: ROC-AUC 0.9868 | PR-AUC 0.5400 | 22.1s
Repair-budget OOF fold 1/5
Repair-budget OOF fold 2/5
Repair-budget OOF fold 3/5
Repair-budget OOF fold 4/5
Repair-budget OOF fold 5/5

OOF prediction generation complete.
Failure probabilities: (56000,)
Repair budgets: (56000,)


### 8. Cost-Based Policy Optimisation

In [7]:
from src.evaluation import tune_policy, calculate_cost

# Find the best inspection threshold and repair-budget multiplier.
policy = tune_policy(
    probabilities=oof_failure_probabilities,
    actual_costs=train_df["repair_cost"],
    predicted_budgets=oof_repair_budgets,
)

best_threshold = policy["threshold"]
best_multiplier = policy["multiplier"]

# Calculate the resulting inspection decisions.
oof_flags = (
    oof_failure_probabilities >= best_threshold
).astype(int)

oof_budgets = np.where(
    oof_flags == 1,
    oof_repair_budgets * best_multiplier,
    0.0,
)

# Independently calculate the selected policy's official cost.
verified_cost = calculate_cost(
    repair_costs=train_df["repair_cost"],
    flags=oof_flags,
    budgets=oof_budgets,
)

print("Selected inspection threshold:", best_threshold)
print("Selected budget multiplier:", best_multiplier)
print(f"OOF policy cost: {verified_cost:,.2f}")
print(f"Vehicles inspected: {oof_flags.sum():,}")
print(f"Inspection rate: {oof_flags.mean():.2%}")

Selected inspection threshold: 0.000303746594408579
Selected budget multiplier: 0.75
OOF policy cost: 59,210.58
Vehicles inspected: 1,278
Inspection rate: 2.28%


### 9. Final Model Training and Test Predictions

In [8]:
from src.feature_analysis import align_test_sensors
from src.repair_model import fit_final_repair_model

# Align test sensor columns with the training data.
test_aligned = align_test_sensors(test_df, train_df)
X_test = test_aligned[features]

# 1. Train final failure classifier on all training data.
final_failure_model = build_final_lgbm()
final_failure_model.fit(X, y)

test_failure_probabilities = final_failure_model.predict_proba(X_test)[:, 1]

# 2. Train final repair-budget regressor on all known failed vehicles.
final_repair_model, final_repair_preprocessor, test_repair_budgets = (
    fit_final_repair_model(
        train_df=train_df,
        test_df=test_aligned,
        features=features,
    )
)

# Validate test predictions.
assert len(test_failure_probabilities) == len(test_df)
assert len(test_repair_budgets) == len(test_df)
assert np.isfinite(test_failure_probabilities).all()
assert np.isfinite(test_repair_budgets).all()
assert ((test_failure_probabilities >= 0) & (test_failure_probabilities <= 1)).all()
assert (test_repair_budgets >= 0).all()

print("Final models trained successfully.")
print("Test failure probabilities:", test_failure_probabilities.shape)
print("Test repair budgets:", test_repair_budgets.shape)

Final models trained successfully.
Test failure probabilities: (20000,)
Test repair budgets: (20000,)


### 10. Submission Generation and Validation

In [9]:
# Apply the policy selected using OOF predictions.
test_flags = (
    test_failure_probabilities >= best_threshold
).astype(int)

test_budgets = np.where(
    test_flags == 1,
    test_repair_budgets * best_multiplier,
    0.0,
)

# Construct submission in the required format.
submission = pd.DataFrame({
    "id": test_df["id"].to_numpy(),
    "flag": test_flags,
    "budget": test_budgets,
})

# Validate submission.
assert len(submission) == len(test_df)
assert submission["id"].is_unique
assert submission["id"].equals(test_df["id"])
assert submission["flag"].isin([0, 1]).all()
assert np.isfinite(submission["budget"]).all()
assert (submission["budget"] >= 0).all()
assert submission.notna().all().all()

# Validate against sample submission.
assert list(submission.columns) == list(sample_submission.columns)
assert submission["id"].equals(sample_submission["id"])

# Save submission.
SUBMISSIONS_DIR.mkdir(parents=True, exist_ok=True)

output_path = SUBMISSIONS_DIR / "SUBMISSION.csv"
submission.to_csv(output_path, index=False)

print("Submission generated successfully!")
print("Output:", output_path)
print(f"Vehicles: {len(submission):,}")
print(f"Inspections: {test_flags.sum():,}")
print(f"Inspection rate: {test_flags.mean():.2%}")
print(f"Mean predicted failure probability: {test_failure_probabilities.mean():.6f}")
print(
    "Mean budget among inspected:",
    f"{test_budgets[test_flags == 1].mean():.2f}"
    if test_flags.sum() else "0.00"
)
print("\nSubmission validation passed.")

Submission generated successfully!
Output: C:\Users\Avnish\Documents\GitHub\fleet_maintenance_p2_team38\submissions\SUBMISSION.csv
Vehicles: 20,000
Inspections: 978
Inspection rate: 4.89%
Mean predicted failure probability: 0.016119
Mean budget among inspected: 400.77

Submission validation passed.


In [10]:
from src.classifier_experiments import run_classifier_experiments

experiment_results, experiment_oof = run_classifier_experiments(
    train_df=train_df,
    features=features,
    folds=folds,
    oof_repair_budgets=oof_repair_budgets,
    baseline_probabilities=oof_failure_probabilities,
)

display(experiment_results)



Running random_forest
Fold 1/5 | ROC-AUC: 0.9865 | PR-AUC: 0.5318 | Time: 3.9s
Fold 2/5 | ROC-AUC: 0.9845 | PR-AUC: 0.4748 | Time: 3.9s
Fold 3/5 | ROC-AUC: 0.9796 | PR-AUC: 0.4190 | Time: 3.7s
Fold 4/5 | ROC-AUC: 0.9879 | PR-AUC: 0.3791 | Time: 3.8s
Fold 5/5 | ROC-AUC: 0.9732 | PR-AUC: 0.4923 | Time: 3.6s
random_forest OOF generation complete.

Running xgboost
Fold 1/5 | ROC-AUC: 0.9901 | PR-AUC: 0.6637 | Time: 6.6s
Fold 2/5 | ROC-AUC: 0.9849 | PR-AUC: 0.5770 | Time: 6.7s
Fold 3/5 | ROC-AUC: 0.9777 | PR-AUC: 0.5170 | Time: 6.5s
Fold 4/5 | ROC-AUC: 0.9900 | PR-AUC: 0.4199 | Time: 6.7s
Fold 5/5 | ROC-AUC: 0.9881 | PR-AUC: 0.5451 | Time: 6.6s
xgboost OOF generation complete.

lightgbm: cost=59,210.58, ROC-AUC=0.9849, PR-AUC=0.5261

random_forest: cost=63,634.49, ROC-AUC=0.9823, PR-AUC=0.4468

xgboost: cost=59,626.46, ROC-AUC=0.9860, PR-AUC=0.5204


,model,roc_auc,pr_auc,oof_cost,threshold,budget_multiplier,inspections,inspection_rate
0,lightgbm,0.984860,0.526133,59210.577681,0.000304,0.75,1278,0.022821
2,xgboost,0.986002,0.520363,59626.462817,0.046665,0.75,1362,0.024321
1,random_forest,0.982298,0.446781,63634.491332,0.052642,0.75,1612,0.028786


In [11]:

# ==========================================
# XGBoost submission candidate
# ==========================================

from src.classifier_experiments import build_experiment_models
from src.preprocessing import fit_preprocessor, transform_features
from src.feature_analysis import align_test_sensors

# Selected from XGBoost OOF policy optimisation
xgb_threshold = 0.046665
xgb_multiplier = 0.75

# Align test sensors exactly as in the baseline pipeline
test_aligned = align_test_sensors(test_df, train_df)

X_train = train_df[features]
X_test = test_aligned[features]
y_train = train_df["failure"].to_numpy()

# Fit preprocessing on the complete training data
xgb_prep = fit_preprocessor(
    X_train,
    imputation="median",
    scale=False,
)

X_train_processed = transform_features(xgb_prep, X_train)
X_test_processed = transform_features(xgb_prep, X_test)

# Train XGBoost on all training data
xgb_final = build_experiment_models(y_train)["xgboost"]
xgb_final.fit(X_train_processed, y_train)

# Predict test failure probabilities
xgb_test_probabilities = xgb_final.predict_proba(
    X_test_processed
)[:, 1]

# Apply OOF-selected policy
xgb_test_flags = (
    xgb_test_probabilities >= xgb_threshold
).astype(int)

xgb_test_budgets = np.where(
    xgb_test_flags == 1,
    test_repair_budgets * xgb_multiplier,
    0.0,
)

# Construct submission
xgb_submission = pd.DataFrame({
    "id": test_df["id"].to_numpy(),
    "flag": xgb_test_flags,
    "budget": xgb_test_budgets,
})

# Validate
assert len(xgb_submission) == len(sample_submission)
assert xgb_submission["id"].equals(sample_submission["id"])
assert xgb_submission["flag"].isin([0, 1]).all()
assert xgb_submission["budget"].notna().all()
assert (xgb_submission["budget"] >= 0).all()

# Save separately; preserve the current submission
xgb_path = SUBMISSIONS_DIR / "SUBMISSION_XGBOOST.csv"
xgb_submission.to_csv(xgb_path, index=False)

print("XGBoost submission generated!")
print("File:", xgb_path)
print("Inspections:", xgb_test_flags.sum())
print("Inspection rate:", f"{xgb_test_flags.mean():.2%}")
print("Mean predicted probability:", xgb_test_probabilities.mean())
print("Mean inspected budget:", 
      xgb_test_budgets[xgb_test_flags == 1].mean())


XGBoost submission generated!
File: C:\Users\Avnish\Documents\GitHub\fleet_maintenance_p2_team38\submissions\SUBMISSION_XGBOOST.csv
Inspections: 1034
Inspection rate: 5.17%
Mean predicted probability: 0.0330224
Mean inspected budget: 393.3051291017363
